In [9]:
try:
    result = 10 / 1 
except ZeroDivisionError as e:
    print("错误",e)
else:
    print('计算成功,结果为',result)
finally:
    print('清理工作完成')

计算成功,结果为 10.0
清理工作完成


In [ ]:
import time
import random
class LLMService:
    def __init__(self):
        self.call_count = 0
    def generate(self,prompt:str) -> str:
        self.call_count += 1
        if random.random() < 0.3:
            raise ConnectionError("API 500: Internal Server Error")
        return f"AI 回复: 关于 '{prompt}' 的深度解析..."
def safe_generate(service:LLMService,prompt:str,max_retries:int = 3) -> str:
    for attempt in range(1,max_retries+1):
        try:
            return service.generate(prompt)
        except ConnectionError as e:
            if attempt == max_retries:
                print(f'重试{max_retries}次后仍失败:{e}')
                raise
            wait_time = 2 ** (attempt - 1)
            print(f'第{attempt}次失败,{wait_time}s后重试...({e})')
            time.sleep(wait_time)
service = LLMService()
try:
    result = safe_generate(service,'什么是RAG?')
    print(f"🎉 成功: {result}")
except ConnectionError:
    print("💀 最终放弃，转入人工处理或降级逻辑")

🎉 成功: AI 回复: 关于 '什么是RAG?' 的深度解析...


In [39]:
import time
import random
class LLMService:
    def __init__(self):
        self.call_count = 0
    def generate(self,prompt:str):
        self.call_count += 1
        if random.random() < 0.9:
            raise ConnectionError("API 500: Internal Server Error")
        return f"AI 回复: 关于 '{prompt}' 的深度解析..."
def safe_generate(service:LLMService,prompt:str,max_retries:int = 3) -> str:
    for attempt in range(1,max_retries+1):
        try:
            return service.generate(prompt)
        except ConnectionError as e:
            if attempt == max_retries:
                print(f"❌ 重试 {max_retries} 次后仍失败: {e}")
                raise
            wait_time = 2 ** (attempt-1)
            print(f"⚠️ 第 {attempt} 次失败，{wait_time}s 后重试... ({e})")
            time.sleep(wait_time)
service = LLMService()
try:
    result = safe_generate(service,'什么是RAG?')
    print(f'成功:{result}')
except ConnectionError:
    print('最终放弃，转入人工处理或降级逻辑')

⚠️ 第 1 次失败，1s 后重试... (API 500: Internal Server Error)
⚠️ 第 2 次失败，2s 后重试... (API 500: Internal Server Error)
成功:AI 回复: 关于 '什么是RAG?' 的深度解析...


In [ ]:
import json

def process_user_data(raw_data: list[str]) -> list[dict]:
    """安全的数据解析管道"""
    valid_users = []
    error_log = []

    for i, raw in enumerate(raw_data):
        try:
            user = json.loads(raw)
            # 业务校验：必须有 name 和 age
            if "name" not in user or not isinstance(user["age"], int):
                raise ValueError(f"字段缺失或类型错误: {user}")
            
            valid_users.append(user)
            
        except json.JSONDecodeError as e:
            error_log.append({"index": i, "error": "JSON格式错误", "detail": str(e)})
        except ValueError as e:
            error_log.append({"index": i, "error": "业务校验失败", "detail": str(e)})
        except Exception as e:
            # 兜底：捕获所有未预料的异常，防止管道崩溃
            error_log.append({"index": i, "error": "未知错误", "detail": repr(e)})

    print(f"📊 处理完成: 成功 {len(valid_users)}, 失败 {len(error_log)}")
    if error_log:
        print(f"📝 错误日志示例: {error_log[:2]}")
    
    return valid_users

# 测试脏数据
dirty_data = [
    '{"name": "Alice", "age": 25}',      # ✅ 正常
    '{invalid json}',                     # ❌ JSON 错误
    '{"name": "Bob"}',                    # ❌ 缺少 age
    '{"name": "Charlie", "age": "30"}',   # ❌ age 类型错误
    '{"name": "David", "age": 40}',       # ✅ 正常
]

process_user_data(dirty_data)

In [41]:
import json
def process_user_data(raw_data:list[str]) -> list[dict]:
    valid_users = []
    error_log = []
    for i,raw in enumerate(raw_data):
        try:
            user = json.loads(raw)
            if "name" not in user or not isinstance(user['age'],int):
                raise ValueError(f"字段缺失或类型错误: {user}")
            valid_users.append(user)
        except json.JSONDecodeError as e:
            error_log.append({"index": i, "error": "JSON格式错误", "detail": str(e)})
        except ValueError as e:
            error_log.append({"index": i, "error": "业务校验失败", "detail": str(e)})
        except Exception as e:
            error_log.append({"index": i, "error": "未知错误", "detail": repr(e)})
    print(f"📊 处理完成: 成功 {len(valid_users)}, 失败 {len(error_log)}")
    if error_log:
        print(f"📝 错误日志示例: {error_log[:2]}")
    return valid_users
dirty_data = [
    '{"name": "Alice", "age": 25}',      # ✅ 正常
    '{invalid json}',                     # ❌ JSON 错误
    '{"name": "Bob"}',                    # ❌ 缺少 age
    '{"name": "Charlie", "age": "30"}',   # ❌ age 类型错误
    '{"name": "David", "age": 40}',       # ✅ 正常
]
process_user_data(dirty_data)

📊 处理完成: 成功 2, 失败 3
📝 错误日志示例: [{'index': 1, 'error': 'JSON格式错误', 'detail': 'Expecting property name enclosed in double quotes: line 1 column 2 (char 1)'}, {'index': 2, 'error': '未知错误', 'detail': "KeyError('age')"}]


[{'name': 'Alice', 'age': 25}, {'name': 'David', 'age': 40}]

In [48]:
import asyncio
class AsyncDatabase:
    async def __aenter__(self):
        print('连接数据库')
        await asyncio.sleep(0.5)
        return self 
    async def __aexit__(self,exc_type,exc_val,exc_tb):
        await asyncio.sleep(0.8)
        return self
async def main():
    async with AsyncDatabase() as db:
        print("📝 执行查询...")
await main()


连接数据库
📝 执行查询...
